# Understanding Seq2Seq — the notebook

The companion notebook to [Understanding Seq2Seq: RNNs, Encoders, Decoders, & Attention](https://yazan.ca/posts/seq2seq/). Read the article first; nothing gets re-explained here. You've seen the ideas, now you build them: a French → English translator, first the classic encoder–decoder, then the attention version, in PyTorch. You also get to see what the article could only describe: the bottleneck breaking a real model on long sentences, and attention finding the alignment on its own.

**How to use this notebook**

- Boilerplate (data wrangling, training loop, plotting) comes prefilled. That part isn't the lesson.
- The important ideas are **exercises**: function skeletons with `TODO(you)` comments saying what goes where. Fill them in.
- A **check cell** after each exercise runs your code and shows what the piece you just built does.
- Each exercise also has a collapsed **✋ Solution** cell (click *Show code* to peek). Try it yourself first. Running the solution overwrites your attempt with a working version, so *Runtime → Run all* works end to end if you'd rather just watch.

Training takes a few minutes per model on a GPU, so pick **Runtime → Change runtime type → T4 GPU** before you start. Everything else is instant.

*Inspired by [tensorflow/nmt](https://github.com/tensorflow/nmt), rebuilt as one PyTorch notebook.*

In [ ]:
# Setup: imports, seed, device. Nothing to fill in here.
import math, random, re, time, unicodedata, urllib.request, zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("running on:", device)
if device.type == "cpu":
    print("tip: Runtime → Change runtime type → T4 GPU makes training ~10x faster")

## 1 · The data *(prefilled — the boring part)*

Sentence pairs from [Tatoeba](https://tatoeba.org) (mirrored by PyTorch). We translate French → English. The article translated into Russian, but the ideas don't care about the language pair, and this dataset is small and downloads reliably.

Same trick as the classic tutorials to keep training fast: only short sentences (under 10 tokens) whose English side starts with simple subjects ("i am", "she is", …). A few minutes of training then gets you real translations.

Three special tokens, all straight from the article:

- `SOS` — the artificial first input that kicks off the decoder (the article's \<START\>)
- `EOS` — the stop word the decoder produces when the sentence is done
- `PAD` — filler so sentences of different lengths can share a batch; the model never gets graded on padding

In [ ]:
# Download the sentence pairs (~5 MB).
DATA_URL = "https://download.pytorch.org/tutorial/data.zip"
if not Path("data/eng-fra.txt").exists():
    urllib.request.urlretrieve(DATA_URL, "data.zip")
    with zipfile.ZipFile("data.zip") as z:
        z.extractall()

first = open("data/eng-fra.txt", encoding="utf-8").readline().strip()
print(f"first line of the file: {first!r}")

In [ ]:
# Vocabulary + cleanup + filtering.
SOS, EOS, PAD = 0, 1, 2    # start-of-sentence, end-of-sentence, padding
MAX_LENGTH = 10            # tokens per sentence, EOS included

class Lang:
    """A vocabulary: word ↔ index, one entry per word the model knows."""
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.index2word = {SOS: "<SOS>", EOS: "<EOS>", PAD: "<PAD>"}
    def add_sentence(self, sentence):
        for word in sentence.split():
            if word not in self.word2index:
                idx = len(self.index2word)
                self.word2index[word] = idx
                self.index2word[idx] = word
    @property
    def n_words(self):
        return len(self.index2word)

def normalize(s):
    s = "".join(c for c in unicodedata.normalize("NFD", s.lower().strip())
                if unicodedata.category(c) != "Mn")     # strip accents: é → e
    s = re.sub(r"([.!?])", r" \1", s)                   # split punctuation into its own token
    return re.sub(r"[^a-zA-Z.!?]+", " ", s).strip()

ENG_PREFIXES = ("i am ", "i m ", "he is ", "he s ", "she is ", "she s ",
                "you are ", "you re ", "we are ", "we re ", "they are ", "they re ")

def load_pairs():
    lines = open("data/eng-fra.txt", encoding="utf-8").read().strip().split("\n")
    both = [[normalize(p) for p in l.split("\t")] for l in lines]
    return [(fra, eng) for eng, fra in both          # flip: we go French → English
            if len(fra.split()) < MAX_LENGTH and len(eng.split()) < MAX_LENGTH
            and eng.startswith(ENG_PREFIXES)]

pairs = load_pairs()
fra_lang, eng_lang = Lang("fra"), Lang("eng")
for fra, eng in pairs:
    fra_lang.add_sentence(fra)
    eng_lang.add_sentence(eng)

print(f"{len(pairs)} sentence pairs, e.g. {random.choice(pairs)}")
print(f"French vocab: {fra_lang.n_words} words · English vocab: {eng_lang.n_words} words")

In [ ]:
# Sentences → tensors of word indices, plus a DataLoader for training.
def sentence_to_ids(lang, sentence, warn_unknown=True):
    ids = []
    for w in sentence.split():
        if w in lang.word2index:
            ids.append(lang.word2index[w])
        elif warn_unknown:
            print(f"(skipping unknown word {w!r})")
    return ids + [EOS]

def sentence_to_tensor(lang, sentence):
    """One sentence → a (1, L) batch of word indices, EOS appended."""
    return torch.tensor([sentence_to_ids(lang, sentence)], device=device)

def make_dataloader(batch_size=32):
    n = len(pairs)
    inp = np.full((n, MAX_LENGTH), PAD, dtype=np.int64)
    tgt = np.full((n, MAX_LENGTH), PAD, dtype=np.int64)
    for i, (fra, eng) in enumerate(pairs):
        f = sentence_to_ids(fra_lang, fra)
        e = sentence_to_ids(eng_lang, eng)
        inp[i, :len(f)] = f
        tgt[i, :len(e)] = e
    data = TensorDataset(torch.from_numpy(inp).to(device), torch.from_numpy(tgt).to(device))
    return DataLoader(data, batch_size=batch_size, shuffle=True)

train_loader = make_dataloader()
xb, yb = next(iter(train_loader))
print("one batch:", tuple(xb.shape), "French id-sequences →", tuple(yb.shape), "English id-sequences")
print("first row (words, then EOS=1, then PAD=2 filler):", xb[0].tolist())

## 2 · An RNN from scratch

The article's whole update rule is one line:

$$h_t = f(W x_t + U h_{t-1})$$

Combine the current word ($x_t$) with the memory so far ($h_{t-1}$), squash, and that's the new memory. Same $W$ and $U$ at every step, one small network applied over and over, and that's what lets it read a sentence of any length.

**Exercise 1.** Implement the step, then the read loop that keeps the whole trail of snapshots. The trail looks like a throwaway detail right now, but attention will dig it back up.

In [ ]:
DIM = 4                            # tiny, so the printouts fit on screen
torch.manual_seed(0)
W = torch.randn(DIM, DIM) * 0.5    # reads the current word
U = torch.randn(DIM, DIM) * 0.5    # reads the memory

def rnn_step(x_t, h_prev):
    # TODO(you): the update rule  h_t = f(W x_t + U h_{t-1})
    #   - W @ x_t    : what the current word contributes
    #   - U @ h_prev : what the memory so far contributes
    #   - f is torch.tanh, squashes the sum into a workable range, usual choice for RNNs but we just need some non linearity
    raise NotImplementedError

def read(words):
    # TODO(you): read the way the encoder reads:
    #   1. start the memory blank: h = torch.zeros(DIM)
    #   2. for each word-vector x in `words`: h = rnn_step(x, h),
    #      and append the new h to `trail` (the snapshots)
    #   3. return trail
    trail = []
    raise NotImplementedError

In [ ]:
#@title ✋ Solution 1 — try it yourself first { display-mode: "form" }
def rnn_step(x_t, h_prev):
    return torch.tanh(W @ x_t + U @ h_prev)

def read(words):
    trail = []
    h = torch.zeros(DIM)
    for x in words:
        h = rnn_step(x, h)
        trail.append(h)
    return trail

In [ ]:
# Check: read a 3-word "sentence", watch the trail of snapshots build up.
torch.manual_seed(1)
sentence = torch.randn(3, DIM)     # stand-ins for the vectors of "I", "love", "you"

trail = read(sentence)
for t, h in enumerate(trail, 1):
    print(f"h{t} = {h.numpy().round(2)}")

print("\nclassic seq2seq keeps only the last snapshot — that's the context:")
print("context =", trail[-1].numpy().round(2))

print("\nand the same two weight matrices read a 7-word sentence too:")
print("context =", read(torch.randn(7, DIM))[-1].numpy().round(2))

That's a real RNN, and everything from here on is this loop dressed up. Plain RNNs are forgetful though, so we'll use `nn.GRU`, one of the gated variants people actually used. Same interface: word-vectors in, trail of snapshots plus final memory out. The article skipped the gates; so will we.

## 3 · The encoder

Embedding, then GRU. The word → vector dictionary from the article is `nn.Embedding`: a trainable table, one row of numbers per French word, looked up by index and trained along with everything else. The forward pass returns both things the article cares about: the whole trail of snapshots (attention will want them later) and the final hidden state, which is actually the context.

**Exercise 2.**

In [ ]:
HIDDEN_SIZE = 128   # the classics used 256, 512, or 1024 hidden units; 128 trains fast and works here

class Encoder(nn.Module):
    """Reads the French sentence; returns every snapshot + the context."""
    def __init__(self, n_words, hidden_size=HIDDEN_SIZE):
        super().__init__()
        # TODO(you): two layers do all the work
        #   self.embedding : the word → vector dictionary, nn.Embedding(n_words, hidden_size)
        #   self.gru       : the reader, nn.GRU(hidden_size, hidden_size, batch_first=True)
        raise NotImplementedError

    def forward(self, word_ids):               # word_ids: (batch, L)
        # TODO(you):
        #   1. look the words up in the embedding          → (batch, L, hidden)
        #   2. run the GRU over them; it returns
        #        snapshots — the whole trail, one memory per word   (batch, L, hidden)
        #        hidden    — the memory after the last word         (1, batch, hidden)
        #   3. return snapshots, hidden      (hidden is the context)
        raise NotImplementedError

In [ ]:
#@title ✋ Solution 2 { display-mode: "form" }
class Encoder(nn.Module):
    """Reads the French sentence; returns every snapshot + the context."""
    def __init__(self, n_words, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.embedding = nn.Embedding(n_words, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, word_ids):
        embedded = self.embedding(word_ids)
        snapshots, hidden = self.gru(embedded)
        return snapshots, hidden

In [ ]:
# Check: push a sentence through an (untrained) encoder, look at the two outputs.
encoder = Encoder(fra_lang.n_words).to(device)

s = "je suis tres fier de vous ."
snapshots, context = encoder(sentence_to_tensor(fra_lang, s))
print(f"{s!r} → {snapshots.shape[1]} tokens read")
print("snapshots:", tuple(snapshots.shape), "— one memory per word (attention will want these)")
print("context:  ", tuple(context.shape), "— the final memory, the only thing classic seq2seq keeps")

## 4 · The decoder's turn

Another RNN, run as a writing loop. Its starting hidden state is the context itself: the decoder picks up exactly where the encoder left off. `SOS` kicks off the first step, every step emits a word, and each word gets fed back in as the next step's input.

Two practical things the article skipped, both standard:

- **Teacher forcing.** During training we feed back the correct word instead of the model's guess. Early guesses are garbage; no point compounding them.
- **Fixed-length loop.** We always run `MAX_LENGTH` steps so a whole batch can move in lockstep, and the loss ignores `PAD` positions. When generating for real, we stop at `EOS`.

**Exercise 3.** One time step, then the loop.

In [ ]:
class Decoder(nn.Module):
    """Writes the English sentence, one word per time step."""
    def __init__(self, n_words, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.embedding = nn.Embedding(n_words, hidden_size)   # English dictionary this time
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, n_words)            # hidden state → a score per English word

    def step(self, word_ids, hidden):          # one time step; word_ids: (batch, 1)
        # TODO(you):
        #   1. embed the incoming word                     → (batch, 1, hidden)
        #   2. one GRU step: self.gru(embedded, hidden)    → output, new_hidden
        #   3. scores = self.out(output)                   → (batch, 1, n_words)
        #   4. return scores, new_hidden
        raise NotImplementedError

    def forward(self, enc_snapshots, context, target=None):
        # The writing loop. `enc_snapshots` is unused here: the classic decoder
        # gets only the context. The attention decoder will use them.
        batch = context.shape[1]
        # TODO(you):
        #   1. the first input is a (batch, 1) tensor of SOS, the article's artificial
        #      first input:  torch.full((batch, 1), SOS, device=device)
        #   2. the first hidden state is the context itself: pick up where the encoder left off
        #   3. loop MAX_LENGTH times:
        #        a. scores, hidden = self.step(word, hidden); collect the scores in a list
        #        b. choose the next input word:
        #             training   (target given) → the correct word, target[:, t:t+1]  ← teacher forcing
        #             generating (no target)    → the model's own pick, scores.argmax(-1)
        #   4. return torch.cat(collected_scores, dim=1), hidden, None
        #      (the None stands in for attention weights; this decoder has none)
        raise NotImplementedError

In [ ]:
#@title ✋ Solution 3 { display-mode: "form" }
class Decoder(nn.Module):
    """Writes the English sentence, one word per time step."""
    def __init__(self, n_words, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.embedding = nn.Embedding(n_words, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, n_words)

    def step(self, word_ids, hidden):
        embedded = self.embedding(word_ids)
        output, hidden = self.gru(embedded, hidden)
        return self.out(output), hidden

    def forward(self, enc_snapshots, context, target=None):
        batch = context.shape[1]
        word = torch.full((batch, 1), SOS, device=device)
        hidden = context
        scores = []
        for t in range(MAX_LENGTH):
            s, hidden = self.step(word, hidden)
            scores.append(s)
            word = target[:, t:t+1] if target is not None else s.argmax(-1)
        return torch.cat(scores, dim=1), hidden, None

In [ ]:
# Check that the whole pipeline runs: encoder, context handoff, decoder.
# (we'll use translate() for every model from here on)
def translate(encoder, decoder, sentence):
    """Greedy decoding: run the model, take the best word each step, stop at EOS."""
    with torch.no_grad():
        snapshots, context = encoder(sentence_to_tensor(fra_lang, sentence))
        scores, _, attn = decoder(snapshots, context)
        words = []
        for idx in scores[0].argmax(-1).tolist():
            if idx == EOS:
                break
            words.append(eng_lang.index2word[idx])
        return " ".join(words), attn

decoder = Decoder(eng_lang.n_words).to(device)
out, _ = translate(encoder, decoder, "je suis heureux .")
print("untrained translator says:", repr(out))
print("\nconfident nonsense: the machinery runs, the weights are random. Time to train.")

## 5 · Training *(prefilled)*

Standard supervised learning, nothing seq2seq-specific: cross-entropy between the decoder's scores and the correct English words (`PAD` positions ignored), Adam on both networks.

In [ ]:
def train(encoder, decoder, n_epochs=40, lr=1e-3):
    enc_opt = torch.optim.Adam(encoder.parameters(), lr=lr)
    dec_opt = torch.optim.Adam(decoder.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)      # padding doesn't count
    losses, t0 = [], time.time()
    for epoch in range(1, n_epochs + 1):
        total = 0.0
        for fra_ids, eng_ids in train_loader:
            enc_opt.zero_grad(); dec_opt.zero_grad()
            snapshots, context = encoder(fra_ids)
            scores, _, _ = decoder(snapshots, context, target=eng_ids)   # teacher forcing
            loss = loss_fn(scores.reshape(-1, scores.shape[-1]), eng_ids.reshape(-1))
            loss.backward()
            enc_opt.step(); dec_opt.step()
            total += loss.item()
        losses.append(total / len(train_loader))
        if epoch == 1 or epoch % 5 == 0:
            print(f"epoch {epoch:3d}/{n_epochs}   loss {losses[-1]:.3f}   ({time.time()-t0:.0f}s)")
    return losses

def plot_losses(*runs):
    plt.figure(figsize=(6, 3))
    for losses, label in runs:
        plt.plot(losses, label=label)
    plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend()
    plt.show()

In [ ]:
# Train the classic model. ~3 min on a T4 GPU; bump N_EPOCHS to 80 for better output.
N_EPOCHS = 40

encoder = Encoder(fra_lang.n_words).to(device)
decoder = Decoder(eng_lang.n_words).to(device)
classic_losses = train(encoder, decoder, N_EPOCHS)
plot_losses((classic_losses, "classic seq2seq"))

In [ ]:
# It translates now. (These come from the training set, so this is the easy test.)
for fra, eng in random.sample(pairs, 5):
    out, _ = translate(encoder, decoder, fra)
    print(f"french : {fra}\n model : {out}\n human : {eng}\n")

## 6 · The bottleneck

Everything the decoder will ever know about the input has to fit through that one fixed-size context vector. A ten word sentence and a fifty word sentence get squeezed into the same amount of space, and the memory fades a little with every update.

We trained on sentences under 10 tokens. Watch what happens as the input grows: the early clauses wash out first.

In [ ]:
# Chain simple clauses with "et" (and) and watch the context vector run out of room.
clauses = ["je suis heureux", "tu es gentille", "il est riche", "nous sommes fatigues"]

bottleneck_tests = [" et ".join(clauses[:n]) + " ." for n in range(1, len(clauses) + 1)]
for s in bottleneck_tests:
    out, _ = translate(encoder, decoder, s)
    print(f"input : {s}\n    → : {out}\n")

## 7 · Attention

The attention model differs from the classic one you just built in two ways:

1. The encoder passes a lot more to the decoder: all of its hidden states, one per input word, instead of just the last one.
2. The decoder gets an extra step before producing each output word, where it decides which of those snapshots matter right now.

The extra step is the same recipe every time: dot products for scores (a snapshot scores high exactly when it points the same way as the decoder's state, because meaning lives in directions), a softmax to turn scores into weights, a weighted sum to build a context vector for this one time step. The article's worked example: scores $0, 0, 2$ softmax into weights $.1, .1, .8$.

**Exercise 4 — the attention step.** Three lines.

In [ ]:
def attention_step(dec_hidden, enc_snapshots):
    """One attention step: score every snapshot, softmax, weighted sum.

    dec_hidden:    (1, batch, hidden) — the decoder's state, doing the scoring
    enc_snapshots: (batch, L, hidden) — every encoder memory, one per input word
    returns:       context (batch, 1, hidden), weights (batch, 1, L)
    """
    query = dec_hidden.permute(1, 0, 2)     # → (batch, 1, hidden), ready for bmm
    # TODO(you):
    #   1. scores — one dot product per snapshot; batched, that's a single matmul:
    #        torch.bmm(query, enc_snapshots.transpose(1, 2))     → (batch, 1, L)
    #   2. weights — softmax the scores over the snapshots:
    #        F.softmax(scores, dim=-1)
    #      positive, sum to 1, and big scores take much bigger shares
    #   3. context — the weighted sum of the snapshots; again one matmul:
    #        torch.bmm(weights, enc_snapshots)                   → (batch, 1, hidden)
    #   return context, weights
    raise NotImplementedError

In [ ]:
#@title ✋ Solution 4 { display-mode: "form" }
def attention_step(dec_hidden, enc_snapshots):
    """One attention step: score every snapshot, softmax, weighted sum.

    dec_hidden:    (1, batch, hidden) — the decoder's state, doing the scoring
    enc_snapshots: (batch, L, hidden) — every encoder memory, one per input word
    returns:       context (batch, 1, hidden), weights (batch, 1, L)
    """
    query = dec_hidden.permute(1, 0, 2)
    scores = torch.bmm(query, enc_snapshots.transpose(1, 2))
    weights = F.softmax(scores, dim=-1)
    context = torch.bmm(weights, enc_snapshots)
    return context, weights

In [ ]:
# Check: the article's example, verbatim. Three snapshots, scores 0, 0, 2.
h1 = torch.tensor([1., 0., 0.])
h2 = torch.tensor([0., 1., 0.])
h3 = torch.tensor([0., 0., 1.])
snapshots = torch.stack([h1, h2, h3])[None]           # (1, 3, 3): batch of one
dec_state = torch.tensor([0., 0., 2.])[None, None]    # (1, 1, 3): the state doing the scoring

print("scores  :", [round(float(h @ dec_state[0, 0]), 1) for h in (h1, h2, h3)])

context, weights = attention_step(dec_state, snapshots)
print("weights :", weights[0, 0].numpy().round(2), "← the article's .1, .1, .8")
print("context :", context[0, 0].numpy().round(2), "= .1·h1 + .1·h2 + .8·h3 (h3 dominates the sum)")

**Exercise 5 — the attention decoder.** The article's numbered recipe for one decoding time step, wired into a module. The writing loop is prefilled, since it's the same loop you already built, just carrying the snapshots along. The whole exercise is `step`: RNN, attention, glue, FFN.

In [ ]:
class AttnDecoder(nn.Module):
    """The decoder again, with the attention step wired in (the article's steps 1–7)."""
    def __init__(self, n_words, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.embedding = nn.Embedding(n_words, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.ffn = nn.Sequential(                      # (5) the feedforward network:
            nn.Linear(hidden_size * 2, hidden_size),   #     takes [h_t ; c_t], glued end to end
            nn.Tanh(),
            nn.Linear(hidden_size, n_words),           # (6) one score per English word
        )

    def step(self, word_ids, hidden, enc_snapshots):
        # TODO(you): the article's steps, one line each
        #   (1) embed the incoming word                                → (batch, 1, hidden)
        #   (2) one GRU step: self.gru(embedded, hidden) → rnn_out, new_hidden
        #       (rnn_out gets tossed, just like the article says;
        #        the actual word comes from the FFN below)
        #   (3) the attention step you just wrote:
        #        attention_step(new_hidden, enc_snapshots) → context, weights
        #   (4) glue hidden state and context end to end:
        #        torch.cat([new_hidden.permute(1, 0, 2), context], dim=-1)
        #   (5)+(6) pass the glued vector through self.ffn → scores
        #   return scores, new_hidden, weights
        raise NotImplementedError

    def forward(self, enc_snapshots, context, target=None):
        # Prefilled: the same writing loop as before. Only two differences:
        # each step also sees the snapshots, and we keep the weights for plotting.
        batch = context.shape[1]
        word = torch.full((batch, 1), SOS, device=device)
        hidden = context                    # (1) start where the encoder left off
        scores, attn = [], []
        for t in range(MAX_LENGTH):
            s, hidden, w = self.step(word, hidden, enc_snapshots)
            scores.append(s); attn.append(w)
            word = target[:, t:t+1] if target is not None else s.argmax(-1)   # (7) feed back, repeat
        return torch.cat(scores, dim=1), hidden, torch.cat(attn, dim=1)

In [ ]:
#@title ✋ Solution 5 { display-mode: "form" }
class AttnDecoder(nn.Module):
    """The decoder again, with the attention step wired in (the article's steps 1–7)."""
    def __init__(self, n_words, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.embedding = nn.Embedding(n_words, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, n_words),
        )

    def step(self, word_ids, hidden, enc_snapshots):
        embedded = self.embedding(word_ids)                            # (1)
        rnn_out, hidden = self.gru(embedded, hidden)                   # (2)
        context, weights = attention_step(hidden, enc_snapshots)       # (3)
        glued = torch.cat([hidden.permute(1, 0, 2), context], dim=-1)  # (4)
        return self.ffn(glued), hidden, weights                        # (5)+(6)

    def forward(self, enc_snapshots, context, target=None):
        batch = context.shape[1]
        word = torch.full((batch, 1), SOS, device=device)
        hidden = context
        scores, attn = [], []
        for t in range(MAX_LENGTH):
            s, hidden, w = self.step(word, hidden, enc_snapshots)
            scores.append(s); attn.append(w)
            word = target[:, t:t+1] if target is not None else s.argmax(-1)
        return torch.cat(scores, dim=1), hidden, torch.cat(attn, dim=1)

In [ ]:
# Train the attention model: same data, same loop. Only the decoder changed.
attn_encoder = Encoder(fra_lang.n_words).to(device)
attn_decoder = AttnDecoder(eng_lang.n_words).to(device)
attn_losses = train(attn_encoder, attn_decoder, N_EPOCHS)
plot_losses((classic_losses, "classic seq2seq"), (attn_losses, "with attention"))

In [ ]:
# The same growing sentences that broke the classic model.
# No single vector the whole sentence has to squeeze through: the decoder gets
# a fresh context at every step, so the early clauses no longer wash out.
for s in bottleneck_tests:
    out_c, _ = translate(encoder, decoder, s)
    out_a, _ = translate(attn_encoder, attn_decoder, s)
    print(f"input     : {s}\nclassic   : {out_c}\nattention : {out_a}\n")

### The alignment falls out of training

Nobody told the model which French word maps to which English word. But the weights are sitting right there, one row per output word. Plot them and see what attention found on its own, crossings included.

In [ ]:
# The attention weights, plotted per output word. Bright = heavily weighted.
def show_alignment(sentence):
    out, attn = translate(attn_encoder, attn_decoder, sentence)
    ids = sentence_to_ids(fra_lang, sentence, warn_unknown=False)
    in_words = [fra_lang.index2word[i] for i in ids]
    out_words = out.split() + ["<EOS>"]
    w = attn[0, :len(out_words), :len(in_words)].cpu().numpy()

    fig, ax = plt.subplots(figsize=(0.6 * len(in_words) + 2, 0.5 * len(out_words) + 1.5))
    ax.matshow(w, cmap="viridis")
    ax.set_xticks(range(len(in_words)), in_words, rotation=45, ha="left")
    ax.set_yticks(range(len(out_words)), out_words)
    ax.set_xlabel("input (French)"); ax.set_ylabel("output (English)")
    plt.show()

show_alignment("je suis tres fier de vous .")
for fra, _ in random.sample(pairs, 2):
    show_alignment(fra)

## 8 · The road to the Transformer

Attention got rid of the bottleneck, but the model underneath is still an RNN: no step can start until the one before it finishes. A fifty word sentence means fifty steps, one after another, and throwing more hardware at it doesn't change that. The attention step is the only part that doesn't have this problem: the scores are just dot products, and none of them waits for any other.

You can measure it:

In [ ]:
# One 50-word sentence: read it with an RNN, then score every pair of words with dot products.
L, H = 50, 256
xs = torch.randn(1, L, H)
gru = nn.GRU(H, H, batch_first=True)

t0 = time.perf_counter()
h = torch.zeros(1, 1, H)
for t in range(L):                    # one word per tick; each step waits for the last
    _, h = gru(xs[:, t:t+1], h)
rnn_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
scores = xs @ xs.transpose(1, 2)      # all 50×50 = 2500 scores, one matmul, no waiting
dot_ms = (time.perf_counter() - t0) * 1000

print(f"50 RNN steps, one after another    : {rnn_ms:7.2f} ms")
print(f"2500 attention scores, all at once : {dot_ms:7.2f} ms")

Which raises a natural question: if the attention step is doing this much of the work, do we even need the RNN? The Transformer's answer is no: throw away the recurrence, keep the attention, and let every word score every other word directly, with the same recipe you just implemented: dot products, softmax, weighted sum.

**Further reading**

- [The article this notebook accompanies](https://yazan.ca/posts/seq2seq/) — the intuition behind everything here
- [tensorflow/nmt](https://github.com/tensorflow/nmt) — the classic NMT tutorial that inspired this notebook
- [Sutskever et al. 2014](https://arxiv.org/abs/1409.3215) — classic seq2seq · [Bahdanau et al. 2014](https://arxiv.org/abs/1409.0473) — attention (the tiny-network scoring) · [Luong et al. 2015](https://arxiv.org/abs/1508.04025) — the dot-product scoring you used
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — where the next post is headed